# Pipeline mestre — Censo 2022 e análise territorial da RMR

Interface de execução do pipeline versionado. A lógica analítica, aquisição, QA e regressão permanece em `src/censo_rmr/`. Nesta versão não existe comando de promoção/sobrescrita dos produtos históricos.

## 0. Montar Drive e instalar a branch de trabalho

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/censo_senso_rmr
!git clone -b agent/pipeline-rmr-v0 https://github.com/hilaliskandar/censo_senso_rmr.git /content/censo_senso_rmr
%cd /content/censo_senso_rmr
!pip -q install -e .


## 1. Escolher o modo

`AUDITAR` é o padrão e não baixa nem recalcula dados. `REPROCESSAR_EM_STAGING` usa cache/IBGE, recalcula os primeiros blocos em `00_Pipeline/02_Regressao` e compara com os produtos históricos.

In [ ]:
MODO = 'AUDITAR'
# Para executar a regressão integral dos primeiros blocos, altere explicitamente para:
# MODO = 'REPROCESSAR_EM_STAGING'


## 2. Executar

In [ ]:
from pathlib import Path
from pprint import pprint
from censo_rmr.pipeline import executar

REPO = Path('/content/censo_senso_rmr')
DRIVE_RAIZ = Path('/content/drive/MyDrive/protocolo his/Censo_2022_Setores_RMR')
assert DRIVE_RAIZ.exists(), f'Pasta do Drive não encontrada: {DRIVE_RAIZ}'

resultado = executar(MODO, REPO, DRIVE_RAIZ)
pprint(resultado)


## 3. Regra de promoção

Mesmo quando todas as comparações retornarem `ok: true`, esta versão apenas registra a equivalência. A promoção dos produtos de staging para as pastas oficiais será implementada como uma etapa separada, condicionada a QA e revisão humana.

In [ ]:
if MODO == 'REPROCESSAR_EM_STAGING':
    print('Equivalência central:', resultado.get('todos_produtos_centrais_equivalentes'))
    print('Promoção permitida nesta versão:', resultado.get('promocao_permitida'))
    print('Relatório detalhado em:', resultado.get('staging'))
